## PyTorch Basics + Custom Training Loop + MLP on MNIST

### PyTorch Fundamentals:
1. **Tensors:**
   - torch.Tensor is a multi-dimensional array stored on CPU or GPU.
   - requires_grad=True lets PyTorch track operations on the tensor to compute gradients.
        ```python
        import torchx = torch.randn(3, 4, requires_grad=True)  # tracked for autograd
        y = (x * 2).sum()
        y.backward()
        print(x.grad)  # dy/dx
        ```
   - Intuition: PyTorch builds a dynamic computation graph during the forward pass and uses it to compute gradients during backward().

2. **Autograd (Automatic Differentiation)**
   - Forward pass: compute predictions (logits) and loss.
   - Backward pass: loss.backward() computes gradients of all parameters.
   - Optimizer step: optimizer.step() updates parameters using gradients.
   - Flow: `forward` → `loss` → `backward` → `optimizer.step()`

3. **nn.Module (Models)**
   - Subclass nn.Module to define layers in __init__ and computation in forward().
   - Compose layers with nn.Sequential for simple stacks.

4. **Data Pipeline (Dataset + DataLoader)**
   - torchvision.datasets.MNIST provides pre-split data.
   - DataLoader handles batching, shuffling, and prefetching.

5. **GPU / Device Handling**
    ```python
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)# Move batches to device during training:
    images, labels = images.to(device), labels.to(device)
    ```

### MLP for MNIST (Custom Training Loop)

1. **Model Architecture (MLP)**
   - `Flatten(28x28)` → `Linear(784→256)` → `ReLU` → `Dropout(0.3)` → `Linear(256→128)` → `ReLU` → `Linear(128→10 logits)`

> Note: For CrossEntropyLoss, do not add Softmax to the model—PyTorch’s loss expects raw logits.

### Why Custom Loops Matter (vs. model.fit())
Keras model.fit() is great for speed and simplicity. PyTorch custom loops give you:
- Fine control over forward/backward passes
- Custom losses, metrics, mixed-precision, gradient clipping
- Research flexibility (e.g., multiple optimizers, GANs, RL)

### Debugging Mindset & Common Pitfalls
- Don’t apply Softmax before nn.CrossEntropyLoss (it expects logits).
- Move data and model to the same device (cuda or cpu).
- Zero gradients each step: optimizer.zero_grad(set_to_none=True).
- Switch modes: model.train() for training; model.eval() with torch.no_grad() for evaluation.
- Check shapes early:
    ```python
    images, labels = next(iter(train_loader))
    print(images.shape, labels.shape)  # e.g., torch.Size([64, 1, 28, 28]) torch.Size([64])
    ```



### Add‑Ons
1. **Learning Rate Scheduler**
    ```python
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1) # call scheduler.step() after each epoch
    ```
2. **Gradient Clipping**
    ```python
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    ```
3. **Mixed Precision (GPU)**
    ```python
    scaler = torch.cuda.amp.GradScaler()
    for images, labels in train_loader:    
        images, labels = images.to(device), labels.to(device)    
        optimizer.zero_grad(set_to_none=True)    
        with torch.cuda.amp.autocast():        
            logits = model(images)        
            loss = criterion(logits, labels)    
            scaler.scale(loss).backward()    
            scaler.step(optimizer)    
            scaler.update()
    ```
